In [1]:
import pandas as pd
import ydata_profiling as yp
candy = pd.read_csv('/Users/A107809368/projects/playground/candy_case_study/data/candy-data.csv')

/Users/A107809368/projects/playground/candy_case_study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
report = yp.ProfileReport(candy)
report.to_file('/Users/A107809368/projects/playground/candy_case_study/reports/eda_report.html')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 10.63it/s]


- The report is saved as an HTML file in the specified directory.
- data is pretty clean, no missing values, no major problems detected at first sight
- only issue at the beginning is that the last variable 'winpercent' has a different format then other percent variables (0-100 instead of 0-1)
- will need to convert that variable before modeling

In [3]:
candy['winpercent'] = candy['winpercent'].apply(lambda x: round(x / 100, 3))

In [4]:
candy.to_csv('/Users/A107809368/projects/playground/candy_case_study/data/candy-data-cleaned.csv', index=False)

In [5]:
candy.sort_values(by='winpercent', ascending=False).head(20)

,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
52,Reese's Peanut Butter cup,1,0,0,1,0,0,0,0,0,0.720,0.651,0.842
51,Reese's Miniatures,1,0,0,1,0,0,0,0,0,0.034,0.279,0.819
79,Twix,1,0,1,0,0,1,0,1,0,0.546,0.906,0.816
28,Kit Kat,1,0,0,0,0,1,0,1,0,0.313,0.511,0.768
64,Snickers,1,0,1,1,1,0,0,1,0,0.546,0.651,0.767
53,Reese's pieces,1,0,0,1,0,0,0,0,1,0.406,0.651,0.734
36,Milky Way,1,0,1,0,1,0,0,1,0,0.604,0.651,0.731
54,Reese's stuffed with pieces,1,0,0,1,0,0,0,0,0,0.988,0.651,0.729
32,Peanut butter M&M's,1,0,0,1,0,0,0,0,1,0.825,0.651,0.715
42,Nestle Butterfinger,1,0,0,1,0,0,0,1,0,0.604,0.767,0.707


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,0.670
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,0.676
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,0.323
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,0.461
4,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,0.523
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Twizzlers,0,1,0,0,0,0,0,0,0,0.220,0.116,0.455
81,Warheads,0,1,0,0,0,0,1,0,0,0.093,0.116,0.390
82,Welch's Fruit Snacks,0,1,0,0,0,0,0,0,1,0.313,0.313,0.444
83,Werther's Original Caramel,0,0,1,0,0,0,1,0,0,0.186,0.267,0.419


In [48]:
test = candy.iloc[:,1:10].apply(lambda x: str(x), axis=1)

In [49]:
import re
combinations = []
for row in test:
    code = re.sub(r'[^\d]', '', row)[:9]
    combinations.append(code)

In [51]:
# count unique elements in the list
unique_combinations = set(combinations)
len(unique_combinations)

29

In [52]:
# count number of occurrences of each unique element in the list
from collections import Counter
combination_counts = Counter(combinations)
combination_counts

Counter({'010000001': 19,
         '010000101': 7,
         '100000001': 6,
         '010000000': 5,
         '010000100': 5,
         '100000010': 4,
         '100100010': 3,
         '000000001': 3,
         '100001010': 3,
         '100100001': 3,
         '100100000': 3,
         '101001010': 2,
         '100010010': 2,
         '000000000': 2,
         '101110010': 2,
         '101000001': 2,
         '101010010': 2,
         '000100001': 1,
         '011000000': 1,
         '101000010': 1,
         '000110010': 1,
         '000000101': 1,
         '101101010': 1,
         '001000001': 1,
         '001000000': 1,
         '110000100': 1,
         '100000000': 1,
         '001000100': 1,
         '100001001': 1})

In [53]:
candy['combination'] = combinations

In [58]:
candy[candy['combination'] == '010000001'][['competitorname', 'winpercent']]

,competitorname,winpercent
11,Chewey Lemonhead Fruit Mix,0.360
12,Chiclets,0.245
13,Dots,0.423
15,Fruit Chews,0.431
18,Haribo Gold Bears,0.571
20,Haribo Sour Bears,0.514
21,Haribo Twin Snakes,0.422
34,Mike & Ike,0.464
44,Nik L Nip,0.224
45,Now & Later,0.394


In [60]:
candy[candy['combination'] == '100100000'][['competitorname', 'winpercent']]

,competitorname,winpercent
51,Reese's Miniatures,0.819
52,Reese's Peanut Butter cup,0.842
54,Reese's stuffed with pieces,0.729


In [61]:
# model winpercent by ols on all other features except for combination and competitorname
import statsmodels.api as sm
X = candy.drop(columns=['winpercent', 'competitorname', 'combination'])
y = candy['winpercent']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             winpercent   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.471
Method:                 Least Squares   F-statistic:                     7.791
Date:                Sun, 02 Nov 2025   Prob (F-statistic):           9.64e-09
Time:                        20:53:30   Log-Likelihood:                 75.756
No. Observations:                  85   AIC:                            -127.5
Df Residuals:                      73   BIC:                            -98.20
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.3453      0.043      7.990      0.000       0.259       0.431
chocolate            0.1976      0.039      5.066      0.000       0.120       0.275
fruity               0.0942      0.038      2.502      0.015       0.019       0.169
caramel              0.0221      0.037      0.605      0.547      -0.051       0.095
peanutyalmondy       0.1006      0.036      2.781      0.007       0.029       0.173
nougat               0.0082      0.057      0.143      0.887      -0.106       0.122
crispedricewafer     0.0892      0.053      1.693      0.095      -0.016       0.194
hard                -0.0616      0.035     -1.782      0.079      -0.131       0.007
bar                  0.0043      0.051      0.085      0.933      -0.097       0.105
pluribus            -0.0086      0.030     -0.284      0.777      -0.069       0.052
sugarpercent         0.0910      0.047      1.953      0.055      -0.002       0.184
pricepercent        -0.0594      0.055     -1.077      0.285      -0.169       0.051
==============================================================================
Omnibus:                        1.037   Durbin-Watson:                   1.728
Prob(Omnibus):                  0.596   Jarque-Bera (JB):                1.009
Skew:                          -0.105   Prob(JB):                        0.604
Kurtosis:                       2.510   Cond. No.                         10.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""